# Olist Order Reviews – Data Preprocessing

## Objective

Prepare the Olist order reviews dataset for the Silver layer.

The preprocessing focuses on data quality validation, text standardization,
data type definition, translation consistency and reproducible quality checks.

## Granularity

One row represents one review record associated with an order.

## Key Definitions

- `id_order_reviews` is generated as a sequential integer starting at 1 and serves as the technical unique identifier of each Silver record.
- `review_id`: business identifier of the review. It is not unique across all records and is therefore not used as the technical unique identifier.
- `order_id`: identifier of the associated order used to establish the relationship with the orders dataset.
- `review_id` + `order_id`: composite business key that uniquely identifies the row-level record in the dataset.

## Processing Steps

1. Load raw data
2. Perform data audit
3. Validate keys and granularity
4. Check missing values and duplicates
5. Standardize Portuguese and English text fields
6. Define explicit data types
7. Validate business rules
8. Run Silver data quality checks
9. Export the Silver dataset

## 1. Setup

Import required Python libraries and define project paths.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RAW_PATH = Path("../datasets/raw/olist_order_reviews_dataset_en.csv")
PROCESSED_PATH = Path("../datasets/processed")

PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

In [3]:
bronze_order_reviews = pd.read_csv(RAW_PATH)


bronze_order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_title_en,review_comment_message,review_comment_message_en,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,NaN,Recebi bem antes do prazo estipulado.,I received it well before the promised deadline.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,NaN,Parabéns lojas lannister adorei comprar pela I...,"Congratulations, Lannister stores! I loved buy...",2018-03-01 00:00:00,2018-03-02 10:26:53


# 2. Data Understanding

Inspect the structure, granularity, data types, and basic characteristics of the order reviews dataset before applying any transformations.

The objective is to understand how reviews are represented in the raw data and identify potential data quality issues that require further investigation.

### 2.1 Data Overview

In [4]:
bronze_order_reviews.shape

(99224, 9)

In [5]:
bronze_order_reviews.head()

,review_id,order_id,review_score,review_comment_title,review_comment_title_en,review_comment_message,review_comment_message_en,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,NaN,NaN,2018-01-18 00:00:00,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,NaN,NaN,2018-03-10 00:00:00,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,NaN,NaN,2018-02-17 00:00:00,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,NaN,Recebi bem antes do prazo estipulado.,I received it well before the promised deadline.,2017-04-21 00:00:00,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,NaN,Parabéns lojas lannister adorei comprar pela I...,"Congratulations, Lannister stores! I loved buy...",2018-03-01 00:00:00,2018-03-02 10:26:53


In [6]:
bronze_order_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   review_id                  99224 non-null  str  
 1   order_id                   99224 non-null  str  
 2   review_score               99224 non-null  int64
 3   review_comment_title       11568 non-null  str  
 4   review_comment_title_en    11564 non-null  str  
 5   review_comment_message     40977 non-null  str  
 6   review_comment_message_en  40947 non-null  str  
 7   review_creation_date       99224 non-null  str  
 8   review_answer_timestamp    99224 non-null  str  
dtypes: int64(1), str(8)
memory usage: 22.5 MB


In [7]:
bronze_order_reviews.describe(include="all")

,review_id,order_id,review_score,review_comment_title,review_comment_title_en,review_comment_message,review_comment_message_en,review_creation_date,review_answer_timestamp
count,99224,99224,99224.000000,11568,11564,40977,40947,99224,99224
unique,98410,98673,NaN,4527,3219,36159,34189,636,98248
top,c444278834184f72b1484dfe47de7f97,c88b1d1b157a9999ce368f218a407141,NaN,Recomendo,Great,Muito bom,Very good,2017-12-19 00:00:00,2017-06-15 23:21:05
freq,3,3,NaN,423,963,230,391,463,4
mean,NaN,NaN,4.086421,NaN,NaN,NaN,NaN,NaN,NaN
std,NaN,NaN,1.347579,NaN,NaN,NaN,NaN,NaN,NaN
min,NaN,NaN,1.000000,NaN,NaN,NaN,NaN,NaN,NaN
25%,NaN,NaN,4.000000,NaN,NaN,NaN,NaN,NaN,NaN
50%,NaN,NaN,5.000000,NaN,NaN,NaN,NaN,NaN,NaN
75%,NaN,NaN,5.000000,NaN,NaN,NaN,NaN,NaN,NaN


### 2.2 Schema & Column Overview

In [8]:
bronze_order_reviews.columns.tolist()

['review_id',
 'order_id',
 'review_score',
 'review_comment_title',
 'review_comment_title_en',
 'review_comment_message',
 'review_comment_message_en',
 'review_creation_date',
 'review_answer_timestamp']

In [9]:

expected_columns = [
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_title_en",
    "review_comment_message",
    "review_comment_message_en",
    "review_creation_date",
    "review_answer_timestamp"
]

assert bronze_order_reviews.columns.tolist() == expected_columns

In [10]:
bronze_order_reviews.dtypes

review_id                      str
order_id                       str
review_score                 int64
review_comment_title           str
review_comment_title_en        str
review_comment_message         str
review_comment_message_en      str
review_creation_date           str
review_answer_timestamp        str
dtype: object

### 2.3 Granularity & Relationships

In [11]:
#review_id
print("Unique review_id:", bronze_order_reviews["review_id"].nunique())
print("Rows with duplicated review_id:", bronze_order_reviews["review_id"].duplicated().sum())

Unique review_id: 98410
Rows with duplicated review_id: 814


In [12]:
#order_id
print("Unique order_id:", bronze_order_reviews["order_id"].nunique())
print("Rows with duplicated order_id:", bronze_order_reviews["order_id"].duplicated().sum())

Unique order_id: 98673
Rows with duplicated order_id: 551


In [13]:
review_order_counts = (
    bronze_order_reviews
    .groupby("review_id")["order_id"]
    .nunique()
    .value_counts()
    .sort_index()
)

review_order_counts

order_id
1    97621
2      764
3       25
Name: count, dtype: int64

In [14]:
order_review_counts = (
    bronze_order_reviews
    .groupby("order_id")["review_id"]
    .nunique()
    .value_counts()
    .sort_index()
)

order_review_counts

review_id
1    98126
2      543
3        4
Name: count, dtype: int64

### 2.4 Review Score Distribution

In [15]:
bronze_order_reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [16]:
bronze_order_reviews["review_score"].describe()

count    99224.000000
mean         4.086421
std          1.347579
min          1.000000
25%          4.000000
50%          5.000000
75%          5.000000
max          5.000000
Name: review_score, dtype: float64

### 2.5 Date & Time Characteristics

In [17]:
bronze_order_reviews[
    ["review_creation_date", "review_answer_timestamp"]
].dtypes

review_creation_date       str
review_answer_timestamp    str
dtype: object

In [18]:
bronze_order_reviews[
    ["review_creation_date", "review_answer_timestamp"]
].head()

,review_creation_date,review_answer_timestamp
0,2018-01-18 00:00:00,2018-01-18 21:46:59
1,2018-03-10 00:00:00,2018-03-11 03:05:13
2,2018-02-17 00:00:00,2018-02-18 14:36:24
3,2017-04-21 00:00:00,2017-04-21 22:02:06
4,2018-03-01 00:00:00,2018-03-02 10:26:53


In [19]:
review_creation_dt = pd.to_datetime(
    bronze_order_reviews["review_creation_date"],
    errors="coerce"
)

review_answer_dt = pd.to_datetime(
    bronze_order_reviews["review_answer_timestamp"],
    errors="coerce"
)

In [20]:
print(
    "Review creation:",
    review_creation_dt.min(),
    "to",
    review_creation_dt.max()
)

print(
    "Review answer:",
    review_answer_dt.min(),
    "to",
    review_answer_dt.max()
)

Review creation: 2016-10-02 00:00:00 to 2018-08-31 00:00:00
Review answer: 2016-10-07 18:32:28 to 2018-10-29 12:27:35


### 2.6 Chapter Summary

The order reviews dataset contains 99,224 records across seven columns covering review identifiers, order identifiers, review scores, review text, and review timestamps.

The analysis shows that review IDs and order IDs are not strictly unique. Most review IDs are associated with one order, while a smaller number are associated with multiple orders. Conversely, most orders contain one review record, while a limited number contain multiple review records. Therefore, neither identifier is sufficient to define a unique row across the complete dataset.

Review scores range from 1 to 5, with higher scores occurring substantially more frequently than lower scores.

The review creation dates cover the period from October 2016 to August 2018, while review answer timestamps extend from October 2016 to October 2018.

The raw dataset stores the timestamp fields as strings. Their values follow a consistent datetime representation and will be evaluated further during the data quality assessment and transformation phase.

Overall, the dataset structure and granularity are sufficiently understood to proceed to a dedicated data quality assessment before applying any transformations.

## 3. Data Quality Assessment

Evaluate the data quality of the order reviews dataset before applying transformations.

The assessment focuses on missing values, duplicate records, valid value ranges, datetime consistency, and text field quality. The objective is to identify issues that require treatment during preprocessing while preserving valid characteristics of the raw dataset.

### 3.1 Missing Value Assessment

In [21]:
missing_summary = (
    bronze_order_reviews.isna()
    .sum()
    .to_frame("missing_count")
)

missing_summary["missing_pct"] = (
    missing_summary["missing_count"]
    / len(bronze_order_reviews)
    * 100
)

missing_summary

,missing_count,missing_pct
review_id,0,0.000000
order_id,0,0.000000
review_score,0,0.000000
review_comment_title,87656,88.341530
review_comment_title_en,87660,88.345562
review_comment_message,58247,58.702532
review_comment_message_en,58277,58.732766
review_creation_date,0,0.000000
review_answer_timestamp,0,0.000000


In [22]:
mandatory_columns = [
    "review_id",
    "order_id",
    "review_score",
    "review_creation_date",
    "review_answer_timestamp"
]

bronze_order_reviews[mandatory_columns].isna().sum()

review_id                  0
order_id                   0
review_score               0
review_creation_date       0
review_answer_timestamp    0
dtype: int64

In [23]:
assert bronze_order_reviews[mandatory_columns].notna().all().all()

In [24]:
comment_columns = [
    "review_comment_title",
    "review_comment_title_en",
    "review_comment_message",
    "review_comment_message_en"
]

bronze_order_reviews[comment_columns].isna().sum()

review_comment_title         87656
review_comment_title_en      87660
review_comment_message       58247
review_comment_message_en    58277
dtype: int64

In [25]:
empty_text_summary = {}

for column in comment_columns:
    empty_text_summary[column] = {
        "empty_string": (bronze_order_reviews[column] == "").sum(),
        "whitespace_only":bronze_order_reviews[column].str.strip().eq("").sum()
    }

empty_text_summary

{'review_comment_title': {'empty_string': np.int64(0),
  'whitespace_only': np.int64(2)},
 'review_comment_title_en': {'empty_string': np.int64(0),
  'whitespace_only': np.int64(0)},
 'review_comment_message': {'empty_string': np.int64(0),
  'whitespace_only': np.int64(27)},
 'review_comment_message_en': {'empty_string': np.int64(0),
  'whitespace_only': np.int64(0)}}

### 3.2 Duplicate Record Validation

In [26]:
exact_duplicates = bronze_order_reviews.duplicated().sum()

exact_duplicates

np.int64(0)

In [27]:
review_id_duplicate_rows = bronze_order_reviews["review_id"].duplicated().sum()

review_id_duplicate_rows

np.int64(814)

In [28]:
order_id_duplicate_rows = bronze_order_reviews["order_id"].duplicated().sum()

order_id_duplicate_rows

np.int64(551)

### 3.3 Value Validation

In [29]:
bronze_order_reviews["review_score"].value_counts().sort_index()

review_score
1    11424
2     3151
3     8179
4    19142
5    57328
Name: count, dtype: int64

In [30]:
assert bronze_order_reviews["review_score"].between(1, 5).all() #kapitel3

### 3.4 Datetime validation

In [31]:
review_creation_dt = pd.to_datetime(
    bronze_order_reviews["review_creation_date"],
    errors="coerce"
)

review_answer_dt = pd.to_datetime(
    bronze_order_reviews["review_answer_timestamp"],
    errors="coerce"
)

print("Invalid review_creation_date:", review_creation_dt.isna().sum())
print("Invalid review_answer_timestamp:", review_answer_dt.isna().sum())

Invalid review_creation_date: 0
Invalid review_answer_timestamp: 0


In [32]:
assert review_creation_dt.notna().all()
assert review_answer_dt.notna().all()

In [33]:
answer_before_creation = review_answer_dt < review_creation_dt

answer_before_creation.sum()

np.int64(0)

In [34]:
assert not answer_before_creation.any()

### 3.5 Text Field Validation

In [35]:
blank_text_summary = {}

for column in comment_columns:
    blank_text_summary[column] = (
        bronze_order_reviews[column].notna()
        & bronze_order_reviews[column].str.strip().eq("")
    ).sum()

blank_text_summary

{'review_comment_title': np.int64(2),
 'review_comment_title_en': np.int64(0),
 'review_comment_message': np.int64(27),
 'review_comment_message_en': np.int64(0)}

### 3.6 Chapter Summary

The data quality assessment identified no missing values in mandatory identifiers, review scores, or timestamp fields.

The review comment fields contain substantial missingness, which is treated as expected because review text is optional. A small number of non-null comment values contain only whitespace and will be normalized during preprocessing.

No exact duplicate rows were identified. Repeated `review_id` and `order_id` values do not represent exact duplicate records and therefore do not require record removal.

All review scores are within the valid range of 1 to 5.

Both timestamp fields can be successfully parsed as datetime values, and no chronological inconsistencies were identified between review creation and answer timestamps.

Overall, no critical data quality issues requiring record-level removal were identified. The main preprocessing requirements are the normalization of text fields and conversion of timestamp fields to appropriate datetime types.

## 4. Data Transformation

Apply the required transformations identified during the data quality assessment to create a clean Silver version of the order reviews dataset.

The transformations focus on text normalization and datetime conversion while preserving the original review content and row-level structure of the dataset.

### 4.1 Create Silver Dataset

In [36]:
silver_order_reviews = bronze_order_reviews.copy()

silver_order_reviews.shape

(99224, 9)

In [37]:
silver_order_reviews["id_order_reviews"] = range(
    1,
    len(silver_order_reviews) + 1
)

silver_order_reviews.insert(
    0,
    "id_order_reviews",
    silver_order_reviews.pop("id_order_reviews")
)

In [38]:
assert silver_order_reviews["id_order_reviews"].is_unique
assert silver_order_reviews["id_order_reviews"].min() == 1
assert silver_order_reviews["id_order_reviews"].max() == len(silver_order_reviews)

### 4.2 Text Normalization

In [39]:
comment_columns = [
    "review_comment_title",
    "review_comment_message"
]

for column in comment_columns:
    silver_order_reviews[column] = (
        silver_order_reviews[column]
        .astype("string")
        .str.lower()
        .str.replace(r"[^a-z\s.,!?]", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )

In [40]:
translation_columns = [
    "review_comment_title_en",
    "review_comment_message_en"
]

for column in translation_columns:
    silver_order_reviews[column] = (
        silver_order_reviews[column]
        .astype("string")
        .str.lower()
        .str.replace(r"[^a-z\s.,!?]", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
        .replace("", pd.NA)
    )

In [41]:
silver_order_reviews.loc[
    silver_order_reviews["review_comment_title"] == "nota dez",
    "review_comment_title_en"
] = "ranking ten"

silver_order_reviews.loc[
    silver_order_reviews["review_comment_title"] == "nota",
    "review_comment_title_en"
] = "ranking"

silver_order_reviews.loc[
    silver_order_reviews["review_comment_message"] == "nota",
    "review_comment_message_en"
] = "ranking"

silver_order_reviews.loc[
    silver_order_reviews["review_comment_title"] == "nenhuma",
    "review_comment_title_en"
] = "none"

silver_order_reviews.loc[
    silver_order_reviews["review_comment_message"] == "nenhuma",
    "review_comment_message_en"
] = "none"

silver_order_reviews.loc[
    silver_order_reviews["review_comment_message"] == "nd",
    "review_comment_message_en"
] = "not available"

silver_order_reviews.loc[
    silver_order_reviews["review_comment_message"] == "no h",
    "review_comment_message_en"
] = "there is no"

In [42]:
for column in comment_columns + translation_columns:
    non_null_values = silver_order_reviews[column].dropna()

    assert (
        non_null_values
        == non_null_values.str.strip()
    ).all()

    assert (
        non_null_values
        == non_null_values.str.lower()
    ).all()

    assert (
        non_null_values
        .str.match(r"^[a-z\s.,!?]+$")
        .all()
    )

In [43]:
assert (
    silver_order_reviews["review_comment_title"].isna()
    == silver_order_reviews["review_comment_title_en"].isna()
).all()

assert (
    silver_order_reviews["review_comment_message"].isna()
    == silver_order_reviews["review_comment_message_en"].isna()
).all()

In [44]:
silver_order_reviews[
    [
        "review_comment_title",
        "review_comment_title_en",
        "review_comment_message",
        "review_comment_message_en"
    ]
].head(10)

,review_comment_title,review_comment_title_en,review_comment_message,review_comment_message_en
0,<NA>,<NA>,<NA>,<NA>
1,<NA>,<NA>,<NA>,<NA>
2,<NA>,<NA>,<NA>,<NA>
3,<NA>,<NA>,recebi bem antes do prazo estipulado.,i received it well before the promised deadline.
4,<NA>,<NA>,parabns lojas lannister adorei comprar pela in...,"congratulations, lannister stores! i loved buy..."
5,<NA>,<NA>,<NA>,<NA>
6,<NA>,<NA>,<NA>,<NA>
7,<NA>,<NA>,<NA>,<NA>
8,<NA>,<NA>,<NA>,<NA>
9,recomendo,i recommend,aparelho eficiente. no site a marca do aparelh...,"efficient device. on the site, the devices bra..."


### 4.3 Datetime Transformation

In [45]:
datetime_columns = [
    "review_creation_date",
    "review_answer_timestamp"
]

for column in datetime_columns:
    silver_order_reviews[column] = pd.to_datetime(
        silver_order_reviews[column],
        errors="raise"
    )

In [46]:
silver_order_reviews[datetime_columns].dtypes

review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

In [47]:
assert silver_order_reviews[datetime_columns].notna().all().all()

In [48]:
answer_before_creation = (
    silver_order_reviews["review_answer_timestamp"]
    < silver_order_reviews["review_creation_date"]
)

assert not answer_before_creation.any()

### 4.4 Transformation Validation

In [49]:
silver_order_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   id_order_reviews           99224 non-null  int64         
 1   review_id                  99224 non-null  str           
 2   order_id                   99224 non-null  str           
 3   review_score               99224 non-null  int64         
 4   review_comment_title       11175 non-null  string        
 5   review_comment_title_en    11175 non-null  string        
 6   review_comment_message     40886 non-null  string        
 7   review_comment_message_en  40886 non-null  string        
 8   review_creation_date       99224 non-null  datetime64[us]
 9   review_answer_timestamp    99224 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(2), str(2), string(4)
memory usage: 19.4 MB


In [50]:
assert len(silver_order_reviews) == len(bronze_order_reviews)
assert silver_order_reviews["review_id"].nunique() == bronze_order_reviews["review_id"].nunique()
assert silver_order_reviews["order_id"].nunique() == bronze_order_reviews["order_id"].nunique()

### 4.5 Silver Dataset Overview

In [51]:
silver_order_reviews.head()

,id_order_reviews,review_id,order_id,review_score,review_comment_title,review_comment_title_en,review_comment_message,review_comment_message_en,review_creation_date,review_answer_timestamp
0,1,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,<NA>,<NA>,<NA>,<NA>,2018-01-18,2018-01-18 21:46:59
1,2,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,<NA>,<NA>,<NA>,<NA>,2018-03-10,2018-03-11 03:05:13
2,3,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,<NA>,<NA>,<NA>,<NA>,2018-02-17,2018-02-18 14:36:24
3,4,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,<NA>,<NA>,recebi bem antes do prazo estipulado.,i received it well before the promised deadline.,2017-04-21,2017-04-21 22:02:06
4,5,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,<NA>,<NA>,parabns lojas lannister adorei comprar pela in...,"congratulations, lannister stores! i loved buy...",2018-03-01,2018-03-02 10:26:53


In [52]:
silver_order_reviews.dtypes

id_order_reviews                      int64
review_id                               str
order_id                                str
review_score                          int64
review_comment_title                 string
review_comment_title_en              string
review_comment_message               string
review_comment_message_en            string
review_creation_date         datetime64[us]
review_answer_timestamp      datetime64[us]
dtype: object

## 4.6 Chapter Summary

The order reviews dataset was transformed into a clean Silver version while preserving the original row-level structure and review identifiers.

The Silver dataset receives a unique key `id_order_reviews` to uniquely identify each review record. The key starts at 1 and preserves the row-level grain of the dataset. The existing `review_id` and `order_id` fields are retained as business identifiers and relationship keys.

The Portuguese review text fields were normalized by converting text to lowercase, removing leading and trailing whitespace, standardizing whitespace, and removing characters outside the defined character set. Empty text values were converted to missing values. The original Portuguese review content was retained.

The English review title and message fields were normalized using the same text transformation rules. Missing English translations for specific rating-related values were completed using defined mappings, while the Portuguese original values remained unchanged. This ensured consistency between the original and translated review fields.

The review creation date and answer timestamp were converted from strings to datetime values. The resulting timestamps remain complete and chronologically consistent.

No records were removed and no identifier values were altered during the transformation.

The resulting Silver dataset preserves the original review information while providing standardized Portuguese and English text fields and appropriately typed timestamp columns. It is now prepared for downstream analysis and integration with the other Olist datasets.

# 5. Silver Validation & Export

Perform the final validation of the transformed order reviews dataset and export the validated Silver dataset for downstream analysis.

The final checks ensure that the dataset structure, data types, row count, identifiers, review scores, text fields, and timestamps meet the defined transformation requirements. The validated Silver dataset is then exported for use in subsequent analysis steps.

### 5.1 Silver Dataset Final Validation

In [53]:
assert len(silver_order_reviews) == len(bronze_order_reviews)

expected_silver_columns = [
    "id_order_reviews",
    "review_id",
    "order_id",
    "review_score",
    "review_comment_title",
    "review_comment_title_en",
    "review_comment_message",
    "review_comment_message_en",
    "review_creation_date",
    "review_answer_timestamp"
]

assert silver_order_reviews.columns.tolist() == expected_silver_columns

assert (
    silver_order_reviews["review_comment_title"].isna()
    == silver_order_reviews["review_comment_title_en"].isna()
).all()

assert (
    silver_order_reviews["review_comment_message"].isna()
    == silver_order_reviews["review_comment_message_en"].isna()
).all()

assert (
    silver_order_reviews["review_comment_title"].notna().sum()
    == silver_order_reviews["review_comment_title_en"].notna().sum()
)

assert (
    silver_order_reviews["review_comment_message"].notna().sum()
    == silver_order_reviews["review_comment_message_en"].notna().sum()
)

assert (
    silver_order_reviews["review_id"].nunique()
    == silver_order_reviews["review_id"].nunique()
)

assert (
    silver_order_reviews["order_id"].nunique()
    == silver_order_reviews["order_id"].nunique()
)

mandatory_columns = [
    "review_id",
    "order_id",
    "review_score",
    "review_creation_date",
    "review_answer_timestamp"
]

assert silver_order_reviews[mandatory_columns].notna().all().all()

assert silver_order_reviews["review_score"].between(1, 5).all()

for column in comment_columns:
    non_null_values = silver_order_reviews[column].dropna()

    assert (
        non_null_values
        == non_null_values.str.strip()
    ).all()

    assert (
        non_null_values
        == non_null_values.str.lower()
    ).all()

    assert (
        non_null_values
        .str.match(r"^[a-z\s.,!?]+$")
        .all()
    )

assert pd.api.types.is_datetime64_any_dtype(
    silver_order_reviews["review_creation_date"]
)

assert pd.api.types.is_datetime64_any_dtype(
    silver_order_reviews["review_answer_timestamp"])

assert silver_order_reviews[
    ["review_creation_date", "review_answer_timestamp"]
].notna().all().all()

assert (
    silver_order_reviews["review_answer_timestamp"]
    >= silver_order_reviews["review_creation_date"]
).all()

print("Silver data validation passed.")
print(f"Rows: {len(silver_order_reviews):,}")
print(f"Unique order_ids: {silver_order_reviews['order_id'].nunique():,}")
print(f"Columns: {len(silver_order_reviews.columns)}")

Silver data validation passed.
Rows: 99,224
Unique order_ids: 98,673
Columns: 10


In [54]:
silver_order_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   id_order_reviews           99224 non-null  int64         
 1   review_id                  99224 non-null  str           
 2   order_id                   99224 non-null  str           
 3   review_score               99224 non-null  int64         
 4   review_comment_title       11175 non-null  string        
 5   review_comment_title_en    11175 non-null  string        
 6   review_comment_message     40886 non-null  string        
 7   review_comment_message_en  40886 non-null  string        
 8   review_creation_date       99224 non-null  datetime64[us]
 9   review_answer_timestamp    99224 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(2), str(2), string(4)
memory usage: 19.4 MB


In [64]:
columns_to_check = [
    "review_comment_title",
    "review_comment_title_en",
    "review_comment_message",
    "review_comment_message_en",
    "review_score"
]

for column in columns_to_check:
    print(f"\n--- {column} ---")

    character_counts = (
        silver_order_reviews[column]
        .dropna()
        .astype(str)
        .str.cat(sep="")
    )

    character_counts = (
        pd.Series(list(character_counts))
        .value_counts()
        .sort_index()
    )

    for character, count in character_counts.items():
        display_character = "<SPACE>" if character == " " else character
        print(f"{display_character}: {count}")


--- review_comment_title ---
<SPACE>: 9794
!: 1029
,: 131
.: 692
?: 46
a: 8062
b: 2364
c: 5415
d: 7138
e: 16791
f: 1202
g: 1451
h: 439
i: 6962
j: 193
k: 193
l: 2460
m: 8228
n: 7043
o: 18733
p: 4360
q: 294
r: 9657
s: 3627
t: 8316
u: 4872
v: 704
w: 33
x: 825
y: 15
z: 313

--- review_comment_title_en ---
<SPACE>: 12088
!: 1035
,: 254
.: 685
?: 48
a: 5715
b: 875
c: 6737
d: 9700
e: 19614
f: 1613
g: 5014
h: 3229
i: 8720
j: 29
k: 640
l: 5690
m: 5897
n: 6588
o: 10980
p: 2819
q: 164
r: 11171
s: 3389
t: 9295
u: 2706
v: 2991
w: 834
x: 877
y: 3125
z: 49

--- review_comment_message ---
<SPACE>: 427808
!: 10550
,: 22485
.: 40521
?: 1161
a: 236423
b: 29835
c: 81000
d: 124507
e: 284716
f: 26224
g: 35300
h: 19748
i: 134458
j: 6899
k: 1970
l: 44548
m: 110036
n: 117117
o: 296574
p: 87373
q: 18007
r: 168793
s: 109594
t: 136559
u: 99477
v: 23977
w: 282
x: 5882
y: 497
z: 13302

--- review_comment_message_en ---
<SPACE>: 497045
!: 10522
,: 30305
.: 53900
?: 1242
a: 165991
b: 34279
c: 89482
d: 151027
e: 3447

In [56]:
length_analysis = []

for column in silver_order_reviews.columns:
    values = silver_order_reviews[column].dropna().astype(str)
    lengths = values.str.len()

    length_analysis.append({
        "column": column,
        "data_type": str(silver_order_reviews[column].dtype),
        "min_length": lengths.min(),
        "max_length": lengths.max(),
        "shortest_value": values.loc[lengths.idxmin()],
        "longest_value": values.loc[lengths.idxmax()]
    })

length_analysis = pd.DataFrame(length_analysis)

length_analysis

,column,data_type,min_length,max_length,shortest_value,longest_value
0,id_order_reviews,int64,1,5,1,10000
1,review_id,str,32,32,7bc2406110b926393aa56f80a40eba40,7bc2406110b926393aa56f80a40eba40
2,order_id,str,32,32,73fc7af87114b39712e6da79b0a377eb,73fc7af87114b39712e6da79b0a377eb
3,review_score,int64,1,1,4,4
4,review_comment_title,string,1,25,x,falta de produto e quebra
5,review_comment_title_en,string,1,45,x,aftersales service leaves a lot to be desired
6,review_comment_message,string,1,204,a,"no recebi o produto, o produto consta como env..."
7,review_comment_message_en,string,1,282,a,i bought several chandeliers and had a small p...
8,review_creation_date,datetime64[us],19,19,2018-01-18 00:00:00,2018-01-18 00:00:00
9,review_answer_timestamp,datetime64[us],19,19,2018-01-18 21:46:59,2018-01-18 21:46:59


### 5.2 Export Validation

In [55]:
output_path = PROCESSED_PATH / "silver_order_reviews_en.csv"

silver_order_reviews.to_csv(
    output_path,
    index=False
)

print(f"Silver dataset exported to: {output_path}")

Silver dataset exported to: ..\datasets\processed\silver_order_reviews_en.csv


## 6. Complete Dataset Summary

The order reviews dataset was processed from the raw source into a validated Silver dataset while preserving the original row-level structure and identifiers.

During the data understanding phase, the dataset structure, granularity, identifiers, review scores, text fields, and timestamp fields were examined. The analysis showed that neither `review_id` nor `order_id` is strictly unique at row level. Therefore, no records were removed based solely on identifier duplication.

The data quality assessment identified substantial missingness in the optional review comment fields but no missing values in mandatory identifiers, review scores, or timestamps. No exact duplicate rows were identified, all review scores were within the valid range of 1 to 5, and the timestamp fields were successfully validated for datetime conversion and chronological consistency.

During transformation, the original Portuguese review fields were normalized by converting text to lowercase, removing leading and trailing whitespace, standardizing whitespace, and restricting characters to the defined character set. The Portuguese review content was retained, including rating-related terms such as `nota` and `nota dez`.

The English review title and message fields were normalized using the same text transformation rules. Missing translations for identified rating-related values were completed using defined mappings, including `nota` → `ranking` and `nota dez` → `ranking ten`. Additional placeholder values were standardized in the English fields where required to maintain consistency between the original and translated fields.

The review creation date and answer timestamp were converted from strings to datetime values. The `review_score` field was retained as an integer because it represents an ordinal rating from 1 to 5.

No records were removed and no identifier values were altered during the transformation process.

The final Silver dataset passed the defined validation checks and was prepared for export as `order_reviews_silver.csv`.

The `order_reviews` dataset is now ready for integration with the other Olist datasets and subsequent business-focused analysis.